In [1]:
# ============================================================
# MODEL PATHS
# ============================================================

NDB_SWIN_PATH = (
    "/kaggle/input/datasets/mahrukh05/"
    "ndb-pretrained-weights/best_stage1_swin.pth"
)

HYBRID_STAGE1_PATH = (
    "/kaggle/input/datasets/mahrukh05/"
    "hybrid-stage-1/best_orchid_hybrid_stage1 (3).pth"
)

HYBRID_STAGE2_PATH = (
    "/kaggle/input/datasets/mahrukh05/"
    "hybrid-stage-2/best_orchid_hybrid_stage2.pth"
)

ORCHID_SWIN_PATH = (
    "/kaggle/input/datasets/mahrukh05/"
    "orchid-swin-model/best_orchid_swin.pth"
)


print("✓ NDB Swin weights :", NDB_SWIN_PATH)
print("✓ Hybrid Stage 1   :", HYBRID_STAGE1_PATH)
print("✓ Hybrid Stage 2   :", HYBRID_STAGE2_PATH)
print("✓ ORCHID Swin      :", ORCHID_SWIN_PATH)

✓ NDB Swin weights : /kaggle/input/datasets/mahrukh05/ndb-pretrained-weights/best_stage1_swin.pth
✓ Hybrid Stage 1   : /kaggle/input/datasets/mahrukh05/hybrid-stage-1/best_orchid_hybrid_stage1 (3).pth
✓ Hybrid Stage 2   : /kaggle/input/datasets/mahrukh05/hybrid-stage-2/best_orchid_hybrid_stage2.pth
✓ ORCHID Swin      : /kaggle/input/datasets/mahrukh05/orchid-swin-model/best_orchid_swin.pth


In [26]:
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00


In [27]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import open_clip
import numpy as np
import gradio as gr

from PIL import Image
from torchvision import transforms


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [28]:
# ============================================================
# CELL 3 — ORCHID CLASS NAMES
# ============================================================

CLASS_NAMES = [
    "mdoscc",
    "normal",
    "osmf",
    "pdoscc",
    "wdoscc"
]

CLASS_DESCRIPTIONS = {
    "mdoscc": "Moderately Differentiated Oral Squamous Cell Carcinoma",
    
    "normal": "Normal Oral Tissue",
    
    "osmf": "Oral Submucous Fibrosis",
    
    "pdoscc": "Poorly Differentiated Oral Squamous Cell Carcinoma",
    
    "wdoscc": "Well Differentiated Oral Squamous Cell Carcinoma"
}

NUM_CLASSES = len(CLASS_NAMES)

print("Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(i, "→", name)

Classes:
0 → mdoscc
1 → normal
2 → osmf
3 → pdoscc
4 → wdoscc


In [29]:
# ============================================================
# CELL 4 — SWIN MULTI-STAGE ENCODER
# ============================================================

class SwinMultiStageEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        # Create base Swin
        swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            num_classes=0
        )

        # IMPORTANT:
        # We directly assign modules so checkpoint keys become:
        #
        # encoder.patch_embed
        # encoder.layers
        # encoder.norm
        #
        # NOT encoder.swin.patch_embed

        self.patch_embed = swin.patch_embed
        self.layers = swin.layers
        self.norm = swin.norm


    def forward(self, x):

        # Patch embedding
        x = self.patch_embed(x)

        # Stage 1
        x = self.layers[0](x)
        s1 = x

        # Stage 2
        x = self.layers[1](x)
        s2 = x

        # Stage 3
        x = self.layers[2](x)
        s3 = x

        # Stage 4
        x = self.layers[3](x)
        s4 = x

        return s1, s2, s3, s4


print("✓ SwinMultiStageEncoder defined")

✓ SwinMultiStageEncoder defined


In [30]:
# ============================================================
# CELL 5 — CLIP PROMPTS
# ============================================================

CLIP_PROMPTS = {
    
    "mdoscc": [
        "a histopathological image of moderately differentiated oral squamous cell carcinoma",
        "a microscopic image of moderately differentiated oral squamous cell carcinoma",
        "a histology slide showing moderately differentiated OSCC",
        "moderately differentiated oral squamous cell carcinoma under a microscope",
        "a pathology image of moderately differentiated oral cancer",
        "oral squamous cell carcinoma with moderate differentiation",
    ],

    "normal": [
        "a histopathological image of normal oral tissue",
        "a microscopic image of normal oral mucosa",
        "a histology image showing healthy oral tissue",
        "a pathology slide of normal oral mucosa",
        "a microscopic view of healthy oral epithelium",
        "normal oral tissue under a microscope",
    ],

    "osmf": [
        "a histopathological image of oral submucous fibrosis",
        "a microscopic image showing oral submucous fibrosis",
        "a histology slide of oral submucous fibrosis",
        "oral mucosa affected by oral submucous fibrosis",
        "a pathology image showing fibrosis of the oral mucosa",
        "a microscopic view of fibrotic oral tissue",
    ],

    "pdoscc": [
        "a histopathological image of poorly differentiated oral squamous cell carcinoma",
        "a microscopic image of poorly differentiated oral squamous cell carcinoma",
        "a histology slide showing poorly differentiated OSCC",
        "poorly differentiated oral squamous cell carcinoma under a microscope",
        "a pathology image of poorly differentiated oral cancer",
        "oral squamous cell carcinoma with poor differentiation",
    ],

    "wdoscc": [
        "a histopathological image of well differentiated oral squamous cell carcinoma",
        "a microscopic image of well differentiated oral squamous cell carcinoma",
        "a histology slide showing well differentiated OSCC",
        "well differentiated oral squamous cell carcinoma",
        "a pathology image of well differentiated oral cancer",
        "oral squamous cell carcinoma with high differentiation",
    ]
}

print("✓ CLIP prompts loaded")

✓ CLIP prompts loaded


In [31]:
# ============================================================
# CELL 6 — LOAD CLIP TEXT ENCODER
# ============================================================

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    model_name="ViT-B-16",
    pretrained="laion2b_s34b_b88k",
    device=device
)

clip_tokenizer = open_clip.get_tokenizer(
    "ViT-B-16"
)

clip_model.eval()

for param in clip_model.parameters():
    param.requires_grad = False


print("✓ CLIP loaded")
print("✓ CLIP frozen")

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

✓ CLIP loaded
✓ CLIP frozen


In [32]:
# ============================================================
# GENERATE CLIP TEXT EMBEDDINGS
# ============================================================

text_embeddings_list = []

with torch.no_grad():

    for class_name in CLASS_NAMES:

        prompts = CLIP_PROMPTS[class_name]

        tokens = clip_tokenizer(
            prompts
        ).to(device)

        embeddings = clip_model.encode_text(
            tokens
        )

        # Normalize
        embeddings = F.normalize(
            embeddings,
            dim=-1
        )

        # Average prompts for each class
        class_embedding = embeddings.mean(
            dim=0
        )

        # Normalize again
        class_embedding = F.normalize(
            class_embedding,
            dim=0
        )

        text_embeddings_list.append(
            class_embedding
        )


text_embeddings = torch.stack(
    text_embeddings_list
)

print("Text embeddings shape:")
print(text_embeddings.shape)

Text embeddings shape:
torch.Size([5, 512])


In [33]:
# ============================================================
# CELL 7 — CROSS ATTENTION FUSION MODULE
# ============================================================

class CrossAttentionFusion(nn.Module):

    def __init__(
        self,
        visual_dim,
        text_dim=512
    ):

        super().__init__()

        self.visual_projection = nn.Linear(
            visual_dim,
            text_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=512,
            num_heads=8,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(
            512
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                512,
                1024
            ),

            nn.GELU(),

            nn.Linear(
                1024,
                512
            )
        )

        self.norm2 = nn.LayerNorm(
            512
        )


    def forward(
        self,
        visual_features,
        text_embeddings
    ):

        # Swin output may be:
        # [B, H, W, C]
        # Convert to sequence

        if visual_features.dim() == 4:

            B, H, W, C = visual_features.shape

            visual_features = visual_features.reshape(
                B,
                H * W,
                C
            )

        # Project visual features to 512
        visual_features = self.visual_projection(
            visual_features
        )

        B = visual_features.shape[0]

        # Expand class text embeddings
        text_features = text_embeddings.unsqueeze(
            0
        ).expand(
            B,
            -1,
            -1
        )

        # Cross attention
        attention_output, _ = self.cross_attention(
            query=visual_features,
            key=text_features,
            value=text_features
        )

        x = self.norm1(
            visual_features + attention_output
        )

        ffn_output = self.ffn(x)

        x = self.norm2(
            x + ffn_output
        )

        return x


print("✓ CrossAttentionFusion defined")

✓ CrossAttentionFusion defined


In [34]:
# ============================================================
# CELL 8 — MULTI-STAGE SEMANTIC FUSION
# ============================================================

class MultiScaleSemanticFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.stage1 = CrossAttentionFusion(
            visual_dim=96
        )

        self.stage2 = CrossAttentionFusion(
            visual_dim=192
        )

        self.stage3 = CrossAttentionFusion(
            visual_dim=384
        )

        self.stage4 = CrossAttentionFusion(
            visual_dim=768
        )


    def forward(
        self,
        s1,
        s2,
        s3,
        s4,
        text_embeddings
    ):

        f1 = self.stage1(
            s1,
            text_embeddings
        )

        f2 = self.stage2(
            s2,
            text_embeddings
        )

        f3 = self.stage3(
            s3,
            text_embeddings
        )

        f4 = self.stage4(
            s4,
            text_embeddings
        )

        return f1, f2, f3, f4


print("✓ MultiScaleSemanticFusion defined")

✓ MultiScaleSemanticFusion defined


In [35]:
# ============================================================
# CELL 9 — RESIDUAL CONVOLUTION BLOCK
# ============================================================

class ResidualConvBlock(nn.Module):

    def __init__(
        self,
        channels=512
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                channels
            ),

            nn.GELU(),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                channels
            )
        )

        self.activation = nn.GELU()


    def forward(self, x):

        residual = x

        x = self.block(x)

        x = x + residual

        x = self.activation(x)

        return x


print("✓ ResidualConvBlock defined")

✓ ResidualConvBlock defined


In [36]:
# ============================================================
# CELL 10 — HYBRID DECODER
# ============================================================

class HybridDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder3 = ResidualConvBlock(
            channels=512
        )

        self.decoder2 = ResidualConvBlock(
            channels=512
        )

        self.decoder1 = ResidualConvBlock(
            channels=512
        )


    def forward(
        self,
        fused_features
    ):

        x = fused_features

        x = self.decoder3(x)

        x = self.decoder2(x)

        x = self.decoder1(x)

        return x


print("✓ HybridDecoder defined")

✓ HybridDecoder defined


In [37]:
# ============================================================
# CELL 11 — FULL SWIN + CLIP HYBRID
# ============================================================

class SwinCLIPHybrid(nn.Module):

    def __init__(
        self,
        text_embeddings,
        num_classes=5
    ):

        super().__init__()

        # ----------------------------------------------------
        # SWIN ENCODER
        # ----------------------------------------------------

        self.encoder = SwinMultiStageEncoder()


        # ----------------------------------------------------
        # CLIP TEXT EMBEDDINGS
        # ----------------------------------------------------

        self.text_embeddings = nn.Parameter(
            text_embeddings.clone(),
            requires_grad=False
        )


        # ----------------------------------------------------
        # MULTI-SCALE FUSION
        # ----------------------------------------------------

        self.semantic_fusion = (
            MultiScaleSemanticFusion()
        )


        # ----------------------------------------------------
        # DECODER
        # ----------------------------------------------------

        self.decoder = HybridDecoder()


        # ----------------------------------------------------
        # CLASSIFIER
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                512,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                0.2
            ),

            nn.Linear(
                256,
                num_classes
            )
        )


    def forward(self, x):

        # ----------------------------------------------------
        # SWIN MULTI-SCALE FEATURES
        # ----------------------------------------------------

        s1, s2, s3, s4 = self.encoder(
            x
        )


        # ----------------------------------------------------
        # SEMANTIC FUSION
        # ----------------------------------------------------

        f1, f2, f3, f4 = (
            self.semantic_fusion(
                s1,
                s2,
                s3,
                s4,
                self.text_embeddings
            )
        )


        # ----------------------------------------------------
        # USE FINAL FUSED FEATURES
        # ----------------------------------------------------

        x = f4


        # ----------------------------------------------------
        # Sequence → Feature Map
        # ----------------------------------------------------

        B, N, C = x.shape

        H = int(N ** 0.5)
        W = H

        x = x.transpose(
            1,
            2
        )

        x = x.reshape(
            B,
            C,
            H,
            W
        )


        # ----------------------------------------------------
        # DECODER
        # ----------------------------------------------------

        x = self.decoder(
            x
        )


        # ----------------------------------------------------
        # GLOBAL POOLING
        # ----------------------------------------------------

        features = F.adaptive_avg_pool2d(
            x,
            1
        )

        features = features.flatten(
            1
        )


        # ----------------------------------------------------
        # CLASSIFICATION
        # ----------------------------------------------------

        logits = self.classifier(
            features
        )


        return {
            "logits": logits,
            "features": features
        }


print("✓ SwinCLIPHybrid defined")

✓ SwinCLIPHybrid defined


In [38]:
# ============================================================
# CELL 12 — CREATE HYBRID MODEL
# ============================================================

hybrid_model = SwinCLIPHybrid(
    text_embeddings=text_embeddings,
    num_classes=NUM_CLASSES
).to(device)


print("✓ Hybrid model created")

print()
print(hybrid_model)

✓ Hybrid model created

SwinCLIPHybrid(
  (encoder): SwinMultiStageEncoder(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    )
    (layers): Sequential(
      (0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=96, out_features=288, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=96, out_features=96, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path1): Identity()
            (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_featur

In [39]:
# ============================================================
# CELL 13 — LOAD ORCHID HYBRID STAGE 2
# ============================================================

checkpoint = torch.load(
    HYBRID_STAGE2_PATH,
    map_location=device,
    weights_only=False
)


print("=" * 70)
print("ORCHID HYBRID STAGE 2 CHECKPOINT")
print("=" * 70)

print(
    "Epoch:",
    checkpoint["epoch"]
)

print(
    "Validation Accuracy:",
    checkpoint["val_accuracy"]
)

print(
    "Validation Loss:",
    checkpoint["val_loss"]
)

ORCHID HYBRID STAGE 2 CHECKPOINT
Epoch: 10
Validation Accuracy: 0.9618717504332756
Validation Loss: 0.13285949573450984


In [40]:
# ============================================================
# FIX / VERIFY CHECKPOINT KEYS
# ============================================================

state_dict = checkpoint[
    "model_state_dict"
]

fixed_state_dict = {}

for key, value in state_dict.items():

    # If old architecture used:
    # encoder.swin.*
    # convert it to:
    # encoder.*

    if key.startswith("encoder.swin."):

        new_key = key.replace(
            "encoder.swin.",
            "encoder.",
            1
        )

    else:

        new_key = key

    fixed_state_dict[new_key] = value


print(
    "Original checkpoint keys:",
    len(state_dict)
)

print(
    "Processed checkpoint keys:",
    len(fixed_state_dict)
)

Original checkpoint keys: 268
Processed checkpoint keys: 268


In [41]:
# ============================================================
# CELL 14 — LOAD WEIGHTS
# ============================================================

load_result = hybrid_model.load_state_dict(
    fixed_state_dict,
    strict=True
)


hybrid_model.eval()


print()
print("=" * 70)
print("✓ ORCHID HYBRID STAGE 2 LOADED SUCCESSFULLY")
print("=" * 70)

print("Model is ready for inference.")


✓ ORCHID HYBRID STAGE 2 LOADED SUCCESSFULLY
Model is ready for inference.


In [42]:
# ============================================================
# CELL 15 — IMAGE PREPROCESSING
# ============================================================

IMG_SIZE = 224


inference_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


print("✓ Image preprocessing ready")

✓ Image preprocessing ready


In [48]:
# ============================================================
# CELL 16 — PREDICTION FUNCTION
# ============================================================

def predict_orchid(image):

    # --------------------------------------------------------
    # Convert image to RGB
    # --------------------------------------------------------

    image = image.convert("RGB")


    # --------------------------------------------------------
    # Preprocess
    # --------------------------------------------------------

    input_tensor = inference_transform(image)

    input_tensor = (
        input_tensor
        .unsqueeze(0)
        .to(device)
    )


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    hybrid_model.eval()


    with torch.no_grad():

        output = hybrid_model(
            input_tensor
        )

        logits = output["logits"]

        probabilities_tensor = torch.softmax(
            logits,
            dim=1
        )[0]


    # --------------------------------------------------------
    # Convert probabilities
    # --------------------------------------------------------

    probabilities = (
        probabilities_tensor
        .detach()
        .cpu()
        .numpy()
    )


    prediction_index = int(
        np.argmax(probabilities)
    )


    predicted_class = CLASS_NAMES[
        prediction_index
    ]


    confidence = float(
        probabilities[
            prediction_index
        ]
    )


    # --------------------------------------------------------
    # All probabilities
    # --------------------------------------------------------

    class_probabilities = {

        CLASS_NAMES[i]: float(
            probabilities[i]
        )

        for i in range(NUM_CLASSES)
    }


    return (
        predicted_class,
        confidence,
        class_probabilities,
        prediction_index
    )


print("✓ Updated prediction function ready")

✓ Updated prediction function ready


In [50]:
!pip install grad-cam -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [51]:
# ============================================================
# CELL 17 — GRAD-CAM ENGINE
# ============================================================

import numpy as np
import torch

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import (
    ClassifierOutputTarget
)

from pytorch_grad_cam.utils.image import (
    show_cam_on_image
)


# ============================================================
# IMAGENET NORMALIZATION
# ============================================================

CAM_MEAN = np.array([
    0.485,
    0.456,
    0.406
])

CAM_STD = np.array([
    0.229,
    0.224,
    0.225
])


# ============================================================
# GRAD-CAM MODEL WRAPPER
# ============================================================

class HybridCAMWrapper(
    torch.nn.Module
):

    def __init__(self, model):

        super().__init__()

        self.model = model


    def forward(self, x):

        output = self.model(x)

        return output["logits"]


# ============================================================
# CREATE WRAPPER
# ============================================================

cam_model = HybridCAMWrapper(
    hybrid_model
).to(device)

cam_model.eval()


# ============================================================
# TARGET LAYER
# ============================================================

target_layer = (
    hybrid_model
    .decoder
    .decoder1
    .block[3]
)


print("✓ Grad-CAM model ready")

print()
print("Target layer:")

print(target_layer)


# ============================================================
# GENERATE GRAD-CAM
# ============================================================

def generate_gradcam(image, prediction_index):


    # --------------------------------------------------------
    # Convert image
    # --------------------------------------------------------

    image = image.convert("RGB")


    # --------------------------------------------------------
    # Preprocess
    # --------------------------------------------------------

    input_tensor = inference_transform(
        image
    )

    input_tensor = (
        input_tensor
        .unsqueeze(0)
        .to(device)
    )


    # --------------------------------------------------------
    # Create Grad-CAM
    # --------------------------------------------------------

    targets = [
        ClassifierOutputTarget(
            prediction_index
        )
    ]


    with GradCAM(

        model=cam_model,

        target_layers=[
            target_layer
        ]

    ) as cam:


        grayscale_cam = cam(

            input_tensor=input_tensor,

            targets=targets

        )[0]


    # --------------------------------------------------------
    # Convert normalized tensor back to RGB
    # --------------------------------------------------------

    rgb_image = (

        input_tensor[0]

        .detach()

        .cpu()

        .numpy()

    )


    rgb_image = np.transpose(

        rgb_image,

        (1, 2, 0)

    )


    # Undo normalization

    rgb_image = (

        rgb_image * CAM_STD

        + CAM_MEAN

    )


    rgb_image = np.clip(

        rgb_image,

        0,

        1

    )


    # --------------------------------------------------------
    # Create heatmap
    # --------------------------------------------------------

    heatmap = show_cam_on_image(

        np.zeros_like(
            rgb_image
        ),

        grayscale_cam,

        use_rgb=True

    )


    # --------------------------------------------------------
    # Create overlay
    # --------------------------------------------------------

    overlay = show_cam_on_image(

        rgb_image.astype(
            np.float32
        ),

        grayscale_cam,

        use_rgb=True

    )


    return (

        (rgb_image * 255)
        .astype(np.uint8),

        heatmap,

        overlay

    )


print("✓ Grad-CAM generator ready")

✓ Grad-CAM model ready

Target layer:
Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
✓ Grad-CAM generator ready


In [44]:
# ============================================================
# CELL 17 — MANUAL TEST
# ============================================================

# Example:
#
# image = Image.open("/kaggle/input/your-image.jpg")
#
# predicted_class, confidence, probabilities = predict_orchid(image)
#
# print("Prediction:", predicted_class)
# print("Confidence:", confidence * 100)
# print(probabilities)

In [54]:
# ============================================================
# CELL 18 — UI PREDICTION + GRAD-CAM
# ============================================================

def ui_predict(image):

    # ========================================================
    # NO IMAGE
    # ========================================================

    if image is None:

        return (
            None,
            None,
            None,
            "### Please upload a histopathology image.",
            {},
            ""
        )

    # ========================================================
    # CONVERT IMAGE
    # ========================================================

    if not isinstance(image, Image.Image):

        image = Image.fromarray(image)

    image = image.convert("RGB")

    # ========================================================
    # PREDICTION
    # ========================================================

    (
        predicted_class,
        confidence,
        probabilities,
        prediction_index
    ) = predict_orchid(image)

    # ========================================================
    # GRAD-CAM
    # ========================================================

    original, heatmap, overlay = generate_gradcam(
        image,
        prediction_index
    )

    # ========================================================
    # RESULT CARD
    # ========================================================

    diagnosis = CLASS_DESCRIPTIONS[
        predicted_class
    ]

    result = f"""
<div class="prediction-card">

<h2>🧠 AI Prediction</h2>

<div class="diagnosis-name">
{predicted_class.upper()}
</div>

<p class="full-diagnosis">
{diagnosis}
</p>

<div class="confidence-box">

<span>Prediction Confidence</span>

<h1>{confidence * 100:.2f}%</h1>

</div>

</div>
"""

    # ========================================================
    # DISCLAIMER
    # ========================================================

    disclaimer = """
<div class="disclaimer">

⚠️ <b>Research Use Only</b><br>

This AI system is intended for research and educational purposes.
It is not designed to replace professional pathological diagnosis
or clinical decision-making.

</div>
"""

    # ========================================================
    # RETURN
    # ========================================================

    return (
        original,
        heatmap,
        overlay,
        result,
        probabilities,
        disclaimer
    )


print("✓ AI + Grad-CAM UI function ready")

✓ AI + Grad-CAM UI function ready


In [ ]:

# ============================================================
# CELL 19 — ORAL AI PRODUCTION UI
# ============================================================

import gradio as gr
import re


# ============================================================
# CLOSE PREVIOUS GRADIO UI
# ============================================================

try:
    demo.close()
    print("✓ Previous UI closed successfully.")

except:
    print("✓ No previous UI was running.")


# ============================================================
# PRODUCTION PREDICTION WRAPPER
# ============================================================

def ui_predict_production(image):

    # ========================================================
    # NO IMAGE CASE
    # ========================================================

    if image is None:

        empty_prediction = """

        <div class="prediction-card">

            <div class="prediction-label">
                🧠 AI PREDICTION
            </div>

            <div class="diagnosis-name">
                READY FOR ANALYSIS
            </div>

            <div class="full-diagnosis">
                Upload a histopathology image and click
                <b>Analyze Image with ORAL AI</b>.
            </div>

        </div>

        """

        return (
            None,
            None,
            None,
            empty_prediction,
            {},
            ""
        )


    # ========================================================
    # CALL EXISTING UI PREDICTION FUNCTION
    # ========================================================

    result = ui_predict(image)


    (
        original_img,
        heatmap_img,
        overlay_img,
        old_prediction_html,
        probabilities,
        disclaimer_html

    ) = result


    # ========================================================
    # SAFETY CHECK
    # ========================================================

    if probabilities is None:

        probabilities = {}


    if not isinstance(probabilities, dict):

        probabilities = {}


    # ========================================================
    # GET PREDICTED CLASS + CONFIDENCE
    # ========================================================

    predicted_class = None
    confidence = None


    if len(probabilities) > 0:

        predicted_class = max(
            probabilities,
            key=probabilities.get
        )

        confidence = float(
            probabilities[predicted_class]
        ) * 100


    # ========================================================
    # FALLBACK FROM PREVIOUS HTML
    # ========================================================

    else:

        if old_prediction_html:

            class_match = re.search(

                r'<div\s+class=["\']diagnosis-name["\']>\s*(.*?)\s*</div>',

                old_prediction_html,

                re.DOTALL

            )


            if class_match:

                predicted_class = (
                    class_match
                    .group(1)
                    .strip()
                )


            confidence_match = re.search(

                r'(\d+(?:\.\d+)?)%',

                old_prediction_html

            )


            if confidence_match:

                confidence = float(
                    confidence_match.group(1)
                )


    # ========================================================
    # FINAL FALLBACK
    # ========================================================

    if not predicted_class:

        predicted_class = "Prediction Unavailable"


    if confidence is None:

        confidence = 0.0


    # ========================================================
    # CLASS NAME MAPPING
    # ========================================================

    class_names = {

        "mdoscc":
        "Moderately Differentiated Oral Squamous Cell Carcinoma",

        "normal":
        "Normal Oral Tissue",

        "osmf":
        "Oral Submucous Fibrosis",

        "pdoscc":
        "Poorly Differentiated Oral Squamous Cell Carcinoma",

        "wdoscc":
        "Well Differentiated Oral Squamous Cell Carcinoma"

    }


    normalized_class = (
        str(predicted_class)
        .lower()
        .strip()
    )


    full_diagnosis = class_names.get(

        normalized_class,

        str(predicted_class)

    )


    display_class = str(
        predicted_class
    ).upper()


    # ========================================================
    # CONFIDENCE INTERPRETATION
    # ========================================================

    if confidence >= 80:

        confidence_status = "High Confidence"

        confidence_message = (
            "The model shows a strong preference "
            "for this classification."
        )

        confidence_class = "high-confidence"


    elif confidence >= 50:

        confidence_status = "Moderate Confidence"

        confidence_message = (
            "The prediction shows moderate separation "
            "from the remaining classes."
        )

        confidence_class = "moderate-confidence"


    else:

        confidence_status = "Low Confidence"

        confidence_message = (
            "Class probabilities are closely distributed. "
            "Consider expert pathological review."
        )

        confidence_class = "low-confidence"


    # ========================================================
    # PROFESSIONAL PREDICTION CARD
    # ========================================================

    prediction_html = f"""

    <div class="prediction-card">

        <div class="prediction-label">

            🧠 AI PREDICTION

        </div>


        <div class="diagnosis-name">

            {display_class}

        </div>


        <div class="full-diagnosis">

            {full_diagnosis}

        </div>


        <div class="confidence-box">

            <div class="confidence-label">

                Prediction Confidence

            </div>


            <div class="confidence-value {confidence_class}">

                {confidence:.2f}%

            </div>

        </div>


        <div class="confidence-status">

            <div class="status-title {confidence_class}">

                ● {confidence_status}

            </div>


            <div class="status-message">

                {confidence_message}

            </div>

        </div>

    </div>

    """


    # ========================================================
    # RETURN EXACTLY 6 OUTPUTS
    # ========================================================

    return (

        original_img,

        heatmap_img,

        overlay_img,

        prediction_html,

        probabilities,

        disclaimer_html

    )


# ============================================================
# CUSTOM CSS — LIGHT MODE ONLY
# ============================================================

CUSTOM_CSS = """

html,
body,
#root {

    margin: 0 !important;
    padding: 0 !important;

    width: 100% !important;

    min-height: 100vh !important;

    background: #f4f7fb !important;

}


.gradio-container {

    width: 100% !important;

    max-width: none !important;

    min-height: 100vh !important;

    margin: 0 !important;

    padding: 0 !important;

    background: #f4f7fb !important;

    color: #1e293b !important;

}


/* =========================================================
   APP WRAPPER
========================================================= */

.app-wrapper {

    width: 100% !important;

    max-width: 1500px !important;

    margin: 0 auto !important;

    padding: 28px 45px 70px 45px !important;

    box-sizing: border-box !important;

}


/* =========================================================
   NAVBAR
========================================================= */

.navbar {

    width: 100%;

    box-sizing: border-box;

    display: flex;

    justify-content: space-between;

    align-items: center;

    padding: 18px 25px;

    border-radius: 20px;

    background:
        linear-gradient(
            135deg,
            #172033,
            #263b5c
        );

    box-shadow:
        0 12px 35px rgba(15,23,42,0.18);

}


.nav-left {

    display: flex;

    align-items: center;

    gap: 14px;

}


.logo-box {

    width: 50px;

    height: 50px;

    display: flex;

    align-items: center;

    justify-content: center;

    border-radius: 15px;

    font-size: 23px;

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #38bdf8
        );

}


.logo-title {

    color: white;

    font-size: 22px;

    font-weight: 700;

    letter-spacing: 1px;

}


.logo-subtitle {

    margin-top: 3px;

    color: #a5b4c8;

    font-size: 11px;

}


.system-status {

    padding: 10px 16px;

    border-radius: 14px;

    color: #cbd5e1;

    font-size: 11px;

    background:
        rgba(255,255,255,0.08);

}


.status-dot {

    display: inline-block;

    width: 7px;

    height: 7px;

    border-radius: 50%;

    background: #22c55e;

    margin-right: 7px;

}


/* =========================================================
   HERO
========================================================= */

.hero-section {

    text-align: center;

    padding:
        70px 20px
        75px 20px;

}


.hero-title {

    font-size: 36px;

    font-weight: 700;

    color: #172033;

}


.hero-description {

    max-width: 800px;

    margin:
        16px auto 0 auto;

    line-height: 1.8;

    font-size: 15px;

    color: #64748b;

}


/* =========================================================
   ANALYSIS ROW
========================================================= */

.analysis-row {

    align-items: flex-start !important;

    gap: 35px !important;

}


/* =========================================================
   SECTION HEADERS
========================================================= */

.section-header {

    height: 72px;

    display: flex;

    align-items: center;

    gap: 13px;

    margin-bottom: 18px;

}


.section-icon {

    width: 45px;

    height: 45px;

    display: flex;

    align-items: center;

    justify-content: center;

    border-radius: 13px;

    background: #eef4ff;

    font-size: 20px;

}


.section-title {

    font-size: 19px;

    font-weight: 700;

    color: #1e293b;

}


.section-subtitle {

    margin-top: 5px;

    font-size: 11px;

    color: #64748b;

}


/* =========================================================
   IMAGE FRAME
========================================================= */

.image-frame {

    border-radius: 18px !important;

    overflow: hidden !important;

    border:
        1px solid #dbe3ef !important;

    background:
        white !important;

    box-shadow:
        0 8px 25px rgba(15,23,42,0.06);

}


/* =========================================================
   ANALYZE BUTTON
========================================================= */

#analyze_button {

    margin-top: 16px;

}


#analyze_button button {

    width: 100% !important;

    height: 54px !important;

    border: none !important;

    border-radius: 14px !important;

    font-size: 14px !important;

    font-weight: 700 !important;

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #38bdf8
        ) !important;

    box-shadow:
        0 10px 25px rgba(37,99,235,0.25) !important;

}


/* =========================================================
   PREDICTION CARD
========================================================= */

.prediction-card {

    min-height: 430px;

    box-sizing: border-box;

    padding: 42px 35px;

    border-radius: 22px;

    text-align: center;

    background:
        linear-gradient(
            145deg,
            #16213a,
            #202f50
        );

    box-shadow:
        0 15px 40px rgba(15,23,42,0.20);

}


.prediction-label {

    color: #a5b4c8;

    font-size: 11px;

    font-weight: 700;

    letter-spacing: 2px;

}


.diagnosis-name {

    margin-top: 25px;

    font-size: 34px;

    font-weight: 800;

    letter-spacing: 2px;

    color: #7dd3fc;

}


.full-diagnosis {

    margin-top: 12px;

    font-size: 14px;

    line-height: 1.6;

    color: #e2e8f0;

}


/* =========================================================
   CONFIDENCE BOX
========================================================= */

.confidence-box {

    margin-top: 28px;

    padding: 18px;

    border-radius: 14px;

    background:
        rgba(15,23,42,0.55);

}


.confidence-label {

    color: #94a3b8;

    font-size: 12px;

}


.confidence-value {

    margin-top: 7px;

    font-size: 38px;

    font-weight: 800;

}


.high-confidence {

    color: #4ade80 !important;

}


.moderate-confidence {

    color: #facc15 !important;

}


.low-confidence {

    color: #fb7185 !important;

}


/* =========================================================
   CONFIDENCE STATUS
========================================================= */

.confidence-status {

    margin-top: 18px;

    padding: 16px;

    border-radius: 14px;

    text-align: left;

    background:
        rgba(255,255,255,0.05);

}


.status-title {

    font-size: 13px;

    font-weight: 700;

}


.status-message {

    margin-top: 7px;

    color: #cbd5e1;

    font-size: 12px;

    line-height: 1.6;

}


/* =========================================================
   PROBABILITY CARD
========================================================= */

.probability-card {

    margin-top: 18px;

    padding: 15px;

    border-radius: 16px;

    background: white;

    border:
        1px solid #dbe3ef;

}


/* =========================================================
   EXPLAINABLE AI
========================================================= */

.xai-header {

    text-align: center;

    margin-top: 90px;

    margin-bottom: 35px;

}


.xai-header h2 {

    font-size: 27px;

    color: #1e293b;

}


.xai-header p {

    max-width: 700px;

    margin: 12px auto;

    color: #64748b;

    font-size: 14px;

    line-height: 1.7;

}


/* =========================================================
   DISCLAIMER
========================================================= */

.disclaimer {

    margin-top: 40px;

    padding: 20px 24px;

    border-radius: 15px;

    background: #fffaf0;

    border-left:
        5px solid #f59e0b;

    color: #92400e;

}


/* =========================================================
   MODEL INFORMATION
========================================================= */

.model-info {

    margin-top: 50px;

    padding: 35px;

    border-radius: 20px;

    background: white;

    border:
        1px solid #dbe3ef;

}


/* =========================================================
   RESPONSIVE DESIGN
========================================================= */

@media (max-width: 900px) {

    .app-wrapper {

        padding: 20px !important;

    }


    .hero-section {

        padding:
            50px 10px !important;

    }


    .hero-title {

        font-size:
            28px !important;

    }

}

"""


# ============================================================
# CREATE GRADIO APPLICATION
# ============================================================

with gr.Blocks(

    title="ORAL AI — Intelligent Oral Histopathology Analysis",

    css=CUSTOM_CSS

) as demo:


    # ========================================================
    # MAIN APP WRAPPER
    # ========================================================

    with gr.Column(
        elem_classes=["app-wrapper"]
    ):


        # ====================================================
        # NAVBAR
        # ====================================================

        gr.HTML(
            """

            <div class="navbar">

                <div class="nav-left">

                    <div class="logo-box">
                        🧬
                    </div>

                    <div>

                        <div class="logo-title">
                            ORAL AI
                        </div>

                        <div class="logo-subtitle">
                            Intelligent Oral Histopathology Analysis Platform
                        </div>

                    </div>

                </div>


                <div class="system-status">

                    <span class="status-dot"></span>

                    AI System Online &nbsp; | &nbsp; Model v1.0

                </div>

            </div>

            """
        )


        # ====================================================
        # HERO SECTION
        # ====================================================

        gr.HTML(
            """

            <div class="hero-section">

                <div class="hero-title">

                    Analyze Oral Histopathology with AI

                </div>


                <div class="hero-description">

                    Upload a histopathology image to receive an
                    AI-assisted classification, confidence analysis,
                    class probability distribution, and visual
                    explanation through Grad-CAM.

                </div>

            </div>

            """
        )


        # ====================================================
        # MAIN ANALYSIS SECTION
        # ====================================================

        with gr.Row(
            elem_classes=["analysis-row"]
        ):


            # =================================================
            # LEFT COLUMN
            # =================================================

            with gr.Column(
                scale=1
            ):


                gr.HTML(
                    """

                    <div class="section-header">

                        <div class="section-icon">
                            🔬
                        </div>

                        <div>

                            <div class="section-title">

                                Histopathology Image

                            </div>

                            <div class="section-subtitle">

                                Upload an oral tissue sample

                            </div>

                        </div>

                    </div>

                    """
                )


                input_image = gr.Image(

                    type="pil",

                    label="Upload Histopathology Image",

                    height=430,

                    elem_classes=["image-frame"]

                )


                predict_button = gr.Button(

                    "✨ Analyze Image with ORAL AI",

                    variant="primary",

                    elem_id="analyze_button"

                )


            # =================================================
            # RIGHT COLUMN
            # =================================================

            with gr.Column(
                scale=1
            ):


                gr.HTML(
                    """

                    <div class="section-header">

                        <div class="section-icon">
                            🧠
                        </div>

                        <div>

                            <div class="section-title">

                                AI Classification Result

                            </div>

                            <div class="section-subtitle">

                                AI-assisted prediction and confidence analysis

                            </div>

                        </div>

                    </div>

                    """
                )


                prediction_output = gr.HTML(

                    value="""

                    <div class="prediction-card">

                        <div class="prediction-label">

                            🧠 AI PREDICTION

                        </div>


                        <div class="diagnosis-name">

                            READY FOR ANALYSIS

                        </div>


                        <div class="full-diagnosis">

                            Upload a histopathology image and run
                            ORAL AI to begin the classification.

                        </div>

                    </div>

                    """

                )


                with gr.Column(
                    elem_classes=["probability-card"]
                ):

                    probability_output = gr.Label(

                        label="📊 Class Probability Distribution",

                        num_top_classes=5

                    )


        # ====================================================
        # EXPLAINABLE AI SECTION
        # ====================================================

        gr.HTML(
            """

            <div class="xai-header">

                <h2>

                    🔍 Explainable AI Analysis

                </h2>


                <p>

                    Explore which regions of the histopathology image
                    contributed most strongly to the AI model's
                    prediction using Grad-CAM visualization.

                </p>

            </div>

            """
        )


        # ====================================================
        # GRAD-CAM OUTPUTS
        # ====================================================

        with gr.Row():


            original_output = gr.Image(

                label="Original Image",

                height=320,

                elem_classes=["image-frame"]

            )


            heatmap_output = gr.Image(

                label="Grad-CAM Attention",

                height=320,

                elem_classes=["image-frame"]

            )


            overlay_output = gr.Image(

                label="Grad-CAM Overlay",

                height=320,

                elem_classes=["image-frame"]

            )


        # ====================================================
        # DISCLAIMER
        # ====================================================

        disclaimer_output = gr.HTML()


        # ====================================================
        # MODEL INFORMATION
        # ====================================================

        with gr.Column(
            elem_classes=["model-info"]
        ):


            gr.Markdown(
                """

                ## 🧠 Model Information


                | Component | Description |
                |---|---|
                | **Visual Backbone** | Swin Transformer |
                | **Semantic Guidance** | CLIP Text Embeddings |
                | **Fusion Strategy** | Multi-Stage Cross Attention |
                | **Decoder** | U-Net Inspired Decoder |
                | **Classifier** | 5-Class Oral Histopathology Classification |


                ### ORCHID Dataset Classes

                - 🔴 **MDOSCC** — Moderately Differentiated OSCC

                - 🟢 **NORMAL** — Normal Oral Tissue

                - 🟣 **OSMF** — Oral Submucous Fibrosis

                - 🟠 **PDOSCC** — Poorly Differentiated OSCC

                - 🔵 **WDOSCC** — Well Differentiated OSCC


                ### Model Performance

                **Test Accuracy:** 95.73%

                **Macro F1-Score:** 96.20%

                **Macro ROC-AUC:** 0.9976

                """
            )


    # ========================================================
    # PREDICTION BUTTON
    # ========================================================

    predict_button.click(

        fn=ui_predict_production,

        inputs=input_image,

        outputs=[

            original_output,

            heatmap_output,

            overlay_output,

            prediction_output,

            probability_output,

            disclaimer_output

        ]

    )


print("ORAL AI Production Interface created successfully")



/tmp/ipykernel_58/2833564274.py:696: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


✓ ORAL AI Production Interface created successfully


In [ ]:
# ============================================================
# CELL 20 — LAUNCH ORAL AI
# ============================================================

demo.launch(
    share=True
)